In [0]:
dbutils.widgets.text("catalogo", "mlops", "Catálogo")
dbutils.widgets.text("esquema", "frecuencia", "Esquema")
dbutils.widgets.text("lote", "1", "Lote")
dbutils.widgets.text("experimento_monitoreo", "/Users/santiagopadu@gmail.com/frecuencia_rc_monitoreo", "Experimento monitoreo")
dbutils.widgets.text("variables_psi", "DrivAge,BonusMalus,VehAge,Density", "Variables PSI")
dbutils.widgets.text("n_tramos_psi", "10", "Tramos PSI")
dbutils.widgets.text("umbral_psi", "0.25", "Umbral PSI")
dbutils.widgets.text("tolerancia_ae", "0.07", "Tolerancia A/E")

catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
lote = int(dbutils.widgets.get("lote"))
experimento_monitoreo = dbutils.widgets.get("experimento_monitoreo")
variables_psi = dbutils.widgets.get("variables_psi").split(",")
n_tramos_psi = int(dbutils.widgets.get("n_tramos_psi"))
umbral_psi = float(dbutils.widgets.get("umbral_psi"))
tolerancia_ae = float(dbutils.widgets.get("tolerancia_ae"))

In [0]:
import numpy as np
import pandas as pd
import mlflow
from pyspark.sql import functions as F

tabla_pred = f"{catalogo}.{esquema}.predicciones"
tabla_nuevas = f"{catalogo}.{esquema}.polizas_nuevas"
tabla_train = f"{catalogo}.{esquema}.polizas_train"

lote_pd = (
    spark.table(tabla_pred).filter(F.col("lote") == lote)
    .join(spark.table(tabla_nuevas).select("IDpol", "ClaimNb"), "IDpol")
    .toPandas()
)
train_pd = spark.table(tabla_train).select(*variables_psi).toPandas()
print("lote", lote, "| pólizas", len(lote_pd))

In [0]:
def psi(referencia, actual, n_tramos):
    cortes = np.unique(np.quantile(referencia, np.linspace(0, 1, n_tramos + 1)))
    cortes[0], cortes[-1] = -np.inf, np.inf
    e = pd.cut(referencia, cortes).value_counts(normalize=True, sort=False).to_numpy()
    a = pd.cut(actual, cortes).value_counts(normalize=True, sort=False).to_numpy()
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

observados = float(lote_pd["ClaimNb"].sum())
esperados = float(lote_pd["siniestros_esperados"].sum())
ae = observados / esperados
psis = {v: psi(train_pd[v], lote_pd[v], n_tramos_psi) for v in variables_psi}

print(f"A/E {ae:.4f} | observados {observados:.0f} | esperados {esperados:.1f}")
print(psis)

In [0]:
alertas_psi = [v for v, p in psis.items() if p > umbral_psi]
alerta_ae = abs(ae - 1) > tolerancia_ae
reentrenar = bool(alertas_psi) or alerta_ae

mlflow.set_experiment(experimento_monitoreo)
with mlflow.start_run(run_name=f"lote_{lote}"):
    mlflow.log_params({
        "lote": lote,
        "version_modelo": int(lote_pd["version_modelo"].iloc[0]),
        "deriva_bonusmalus": int(lote_pd["deriva_bonusmalus"].iloc[0]),
        "umbral_psi": umbral_psi,
        "tolerancia_ae": tolerancia_ae,
    })
    mlflow.log_metrics({"ae": ae, "observados": observados, "esperados": esperados, **{f"psi_{v}": p for v, p in psis.items()}})
    mlflow.set_tag("reentrenar", str(reentrenar).lower())

dbutils.jobs.taskValues.set(key="reentrenar", value=reentrenar)
print("reentrenar:", reentrenar, "| PSI en alerta:", alertas_psi, "| alerta A/E:", alerta_ae)